# 🧠 Model Explainability using SHAP

## Enterprise AutoML Platform

## 📖 About this Notebook

This notebook demonstrates Explainable AI using SHAP.

The notebook explains both global and local model predictions, making the machine learning model more interpretable and transparent.

Sections

1. Title
2. Objective
3. Import Libraries
4. Load Dataset
5. Data Preprocessing
6. Train XGBoost Model
7. Create SHAP Explainer
8. SHAP Summary Plot
9. SHAP Bar Plot
10. Force Plot
11. Waterfall Plot
12. Feature Importance
13. Save Reports
14. Conclusion

### Objective

This notebook demonstrates Explainable Artificial Intelligence (XAI) using SHAP.

Topics Covered

- SHAP Values
- Feature Importance
- SHAP Summary Plot
- Force Plot
- Waterfall Plot
- Explain Individual Predictions

Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import shap
import joblib
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from sklearn.model_selection import train_test_split

from xgboost import XGBClassifier

Load Dataset

In [ ]:
df = pd.read_csv(
    "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

Encode Target

df["Churn"] = df["Churn"].map(
    {
        "No":0,
        "Yes":1,
    }
)

Features

In [ ]:
X = df.drop(
    columns=["Churn"]
)

y = df["Churn"]

Features

In [ ]:
X = df.drop(
    columns=["Churn"]
)

y = df["Churn"]

Numerical Columns

In [ ]:
numerical_columns = X.select_dtypes(
    include=["int64","float64"]
).columns.tolist()

Categorical Columns

In [ ]:
categorical_columns = X.select_dtypes(
    include=["object"]
).columns.tolist()

Train Test Split

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

Build Preprocessor

In [ ]:
preprocessor = ColumnTransformer(
    [
        (
            "num",
            Pipeline(
                [
                    (
                        "imputer",
                        SimpleImputer(strategy="median"),
                    ),
                    (
                        "scaler",
                        StandardScaler(),
                    ),
                ]
            ),
            numerical_columns,
        ),
        (
            "cat",
            Pipeline(
                [
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent",
                        ),
                    ),
                    (
                        "encoder",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=False,
                        ),
                    ),
                ]
            ),
            categorical_columns,
        ),
    ]
)

X_train = preprocessor.fit_transform(
    X_train
)

X_test = preprocessor.transform(
    X_test
)

Train Model

In [ ]:
model = XGBClassifier(
    eval_metric="logloss",
    random_state=42,
)

model.fit(
    X_train,
    y_train,
)

SHAP Explainer

In [ ]:
explainer = shap.TreeExplainer(
    model
)

shap_values = explainer.shap_values(
    X_test
)

Summary Plot

In [ ]:
shap.summary_plot(
    shap_values,
    X_test,
    feature_names=preprocessor.get_feature_names_out(),
)

Bar Plot

In [ ]:
shap.summary_plot(
    shap_values,
    X_test,
    feature_names=preprocessor.get_feature_names_out(),
    plot_type="bar",
)

Force Plot

In [ ]:
shap.initjs()

shap.force_plot(
    explainer.expected_value,
    shap_values[0],
    X_test[0],
    feature_names=preprocessor.get_feature_names_out(),
)

Waterfall Plot

In [ ]:
explanation = explainer(
    X_test
)

shap.plots.waterfall(
    explanation[0]
)

Feature Importance Table

In [ ]:
importance = pd.DataFrame(
    {
        "Feature":
        preprocessor.get_feature_names_out(),

        "Importance":
        abs(shap_values).mean(axis=0),
    }
)

importance = importance.sort_values(
    by="Importance",
    ascending=False,
)

importance.head(20)

Save Feature Importance

In [ ]:
importance.to_csv(
    "../reports/feature_importance.csv",
    index=False,
)

Save Summary Plot

In [ ]:
plt.figure()

shap.summary_plot(
    shap_values,
    X_test,
    feature_names=preprocessor.get_feature_names_out(),
    show=False,
)

plt.savefig(
    "../reports/shap_summary.png",
    dpi=300,
    bbox_inches="tight",
)

Save Bar Plot

In [ ]:
plt.figure()

shap.summary_plot(
    shap_values,
    X_test,
    feature_names=preprocessor.get_feature_names_out(),
    plot_type="bar",
    show=False,
)

plt.savefig(
    "../reports/feature_importance.png",
    dpi=300,
    bbox_inches="tight",
)

Save Model

In [ ]:
joblib.dump(
    model,
    "../saved_models/xgboost_model.pkl",
)

# 📌 Key Insights

The SHAP analysis provides feature-level explanations for model predictions.

Key observations:

- Feature importance ranking
- Global model explainability
- Individual prediction explanations
- Business interpretation of predictions

# ✅ Conclusion

Successfully completed

- SHAP Explainability
- Summary Plot
- Feature Importance
- Force Plot
- Waterfall Plot
- Feature Importance CSV
- Saved Visual Reports

The trained model is now fully explainable and ready for production deployment.

---

# 🏭 Production Implementation

The Enterprise AutoML Platform automatically generates explainability reports after model training.

Generated artifacts include:

- SHAP Summary Plot
- Feature Importance Plot
- Feature Importance CSV
- Explainability Reports

These reports are stored automatically for further analysis.